In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import re
import time
import pickle
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score
from xgboost import XGBClassifier

MINIMAX_API_KEY  = ""
MINIMAX_URL      = "https://api.minimaxi.com/v1/chat/completions"
SAVE_PATH_TEST   = os.path.expanduser("~/Desktop/text_analyst_results_test_matched_sample1.pkl")
SAVE_PATH_TRAIN  = os.path.expanduser("~/Desktop/text_analyst_results_train_matched_sample1.pkl")

# Logit / sigmoid helpers (used in logit-space fusion)
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=float)))


# Generic LLM call helper (used by green_llm_review and other agents)
def call_llm(prompt, max_tokens=300, temperature=0.2):
    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": max_tokens, "temperature": temperature},
        timeout=30,
    )
    return resp.json()['choices'][0]['message']['content']


## 1. Data Loading

In [ ]:
df = pd.read_csv('/Users/ljw/Desktop/loan_default.csv', low_memory=False)

# label (0/1) and desc are already present in the cleaned CSV
print(f"Loaded: {len(df)} records, default rate: {df['label'].mean():.1%}")
print(f"Grade distribution:")
print(df['grade'].value_counts().sort_index())

## 2. Train / Test Split

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Train: {len(train)}, default rate: {train['label'].mean():.1%}")
print(f"Test:  {len(test)},  default rate: {test['label'].mean():.1%}")

## 3. Numeric Feature Definitions

In [ ]:
NUM_FEATURES = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc',
    'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies'
]

## 4. White Agent — Text Analyst

In [ ]:
def text_analyst_agent(desc_text: str) -> dict:
    """Score borrower description; returns continuous text_risk_score (0-1)."""
    desc_text = re.sub(r'<[^>]+>', ' ', str(desc_text)).strip()[:500]

    word_count  = len(desc_text.split())
    has_numbers = bool(re.search(r'\$[\d,]+|\d+%|\d+ months?|\d+ years?', desc_text))
    has_plan    = bool(re.search(r'will pay|plan to|intend|commit|currently|stable', desc_text, re.I))
    has_stress  = bool(re.search(r'emergency|urgent|desperate|behind|struggling|need immediately', desc_text, re.I))

    prompt = (
        "You are evaluating loan applications from Grade C-G borrowers on LendingClub.\n"
        "These borrowers already have elevated credit risk — roughly 15-25% will default.\n"
        "Your text_risk_score should reflect this: a TYPICAL C-G borrower with a vague description "
        "should score around 0.40-0.50, not 0.20-0.30.\n\n"
        f"Description: {desc_text}\n\n"
        "Scoring anchors for text_risk_score:\n"
        "  0.05-0.20: unusually strong evidence — specific dollar amounts, concrete timeline, "
        "verified stable income, explicit low-rate refinancing with repayment math\n"
        "  0.20-0.35: genuinely positive signals — mentions stable job, clear single purpose, "
        "no stress language, some concrete detail\n"
        "  0.35-0.55: average C-G borrower — vague purpose, standard language, "
        "no strong signals in either direction; debt consolidation alone falls here\n"
        "  0.55-0.70: mild concern — multiple debts mentioned without plan, "
        "vague about income, hints of urgency or pressure\n"
        "  0.70-0.90: clear stress signals — urgent tone, behind on bills, "
        "no income mentioned, consolidating multiple problem debts with no plan\n"
        "  0.90-1.00: severe distress — explicitly behind on payments, emergency cash need, "
        "no repayment plan whatsoever\n\n"
        "IMPORTANT: Do NOT default to 0.2-0.3 for all descriptions. "
        "If you are uncertain, score 0.40-0.50. "
        "Debt consolidation by itself (without concrete repayment details) is 0.40-0.50, not 0.20-0.30.\n\n"
        "Output a single JSON object and nothing else:\n"
        '{"text_risk_score": <float 0.0-1.0>,\n'
        ' "confidence": <float 0.0-1.0>,\n'
        ' "risk_signals": [<up to 3 quoted phrases>],\n'
        ' "protective_signals": [<up to 3 quoted phrases>],\n'
        ' "reasoning": "<one sentence>"}\n\n'
        "Confidence guide:\n"
        "  0.85-0.95: specific dollar amounts, timelines, or explicit repayment plans\n"
        "  0.65-0.84: some concrete details\n"
        "  0.50-0.64: vague or only 1-2 sentences\n"
        "  0.40-0.49: extremely short or nearly no information"
    )

    for attempt in range(3):
        try:
            resp = requests.post(
                MINIMAX_URL,
                headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
                json={"model": "MiniMax-Text-01",
                      "messages": [{"role": "user", "content": prompt}],
                      "max_tokens": 300, "temperature": 0.3},
                timeout=30
            )
            raw = resp.json()['choices'][0]['message']['content']
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if not match:
                raise ValueError(f"No JSON: {raw[:100]}")
            result = json.loads(match.group(0))

            # Calibrate confidence by text quality
            quality  = (min(word_count / 100, 0.3) + (0.2 if has_numbers else 0)
                        + (0.15 if has_plan else 0) + (0.1 if has_stress else 0))
            raw_conf = result.get('confidence', 0.5)
            result['confidence'] = round(min(max(0.6 * raw_conf + 0.4 * (0.4 + quality), 0.4), 0.95), 3)
            result['text_risk_score'] = round(min(max(float(result.get('text_risk_score', 0.45)), 0.0), 1.0), 3)
            return result
        except Exception as e:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

## 5. Run Text Analysis (Grades C–G)

In [ ]:
def run_text_analysis(df_subset, save_path):
    """Run text_analyst_agent on df_subset with resume support."""
    try:
        with open(save_path, 'rb') as f:
            results = pickle.load(f)
        done_idx  = {r['idx'] for r in results}
        remaining = len(df_subset) - len(done_idx)
        print(f"  Resuming {os.path.basename(save_path)}: {len(results)} done, {remaining} remaining")
    except FileNotFoundError:
        results, done_idx = [], set()
        print(f"  Starting fresh: {len(df_subset)} records")

    errors = []
    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
        if idx in done_idx:
            continue
        try:
            result = text_analyst_agent(row['desc'])
            result['idx'] = idx
            results.append(result)
            if len(results) % 50 == 0:
                with open(save_path, 'wb') as f:
                    pickle.dump(results, f)
        except Exception as e:
            errors.append({'idx': idx, 'error': str(e)})
            time.sleep(1)
        time.sleep(0.1)

    with open(save_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"  Done: {len(results)}, failed: {len(errors)}")
    return results


# C-G grades only; cap API calls with TEXT_SAMPLE_SIZE
TEXT_SAMPLE_SIZE = 2000  # increase for a production run

cg_test  = test[test['grade'].isin(['C', 'D', 'E', 'F', 'G'])]
cg_train = train[train['grade'].isin(['C', 'D', 'E', 'F', 'G'])]

target_test  = (cg_test.sample(n=min(TEXT_SAMPLE_SIZE // 4, len(cg_test)),   random_state=42)
                if len(cg_test)  > TEXT_SAMPLE_SIZE // 4  else cg_test).copy()
target_train = (cg_train.sample(n=min(TEXT_SAMPLE_SIZE,       len(cg_train)), random_state=42)
                if len(cg_train) > TEXT_SAMPLE_SIZE         else cg_train).copy()

print(f"C-G train pool: {len(cg_train)}  →  text-analyze: {len(target_train)}")
print(f"C-G test  pool: {len(cg_test)}   →  text-analyze: {len(target_test)}")

print("\n--- Running on TEST ---")
run_text_analysis(target_test, SAVE_PATH_TEST)
print("\n--- Running on TRAIN ---")
run_text_analysis(target_train, SAVE_PATH_TRAIN)

## 6. Build Fused Subset DataFrame

In [112]:
def load_text_subset(df, save_path):
    with open(save_path, 'rb') as f:
        results = pickle.load(f)
    rdf = pd.DataFrame(results).set_index('idx')

    # Backward compat: old format used 'risk_label' (binary)
    if 'text_risk_score' not in rdf.columns:
        rdf['text_risk_score'] = rdf['risk_label'].astype(float)

    sub = df.join(rdf[['text_risk_score', 'confidence']], how='left')
    sub = sub.rename(columns={'confidence': 'text_confidence'})
    return sub.dropna(subset=['text_risk_score', 'text_confidence'])


target_test  = test[test['grade'].isin(['C', 'D', 'E', 'F', 'G'])].copy()
target_train = train[train['grade'].isin(['C', 'D', 'E', 'F', 'G'])].copy()

subset_train = load_text_subset(target_train, SAVE_PATH_TRAIN)
subset_test  = load_text_subset(target_test,  SAVE_PATH_TEST)

print(f"subset_train: {len(subset_train)} records, default rate: {subset_train['label'].mean():.2f}")
print(f"subset_test:  {len(subset_test)}  records, default rate: {subset_test['label'].mean():.2f}")
print(f"Raw text_risk_score (train): mean={subset_train['text_risk_score'].mean():.3f}, "
      f"std={subset_train['text_risk_score'].std():.3f}")

# ── Per-grade z-score normalization (train stats → applied to both) ─────────
# Linear stretch fails because LLM underestimates absolute risk levels.
# Z-score normalization uses rank signal within each grade — if the LLM
# correctly orders borrowers by relative risk, this preserves that signal
# while re-centering each grade at 0.5 (neutral).
grade_norm_stats = {}
for grade, grp in subset_train.groupby('grade'):
    m = grp['text_risk_score'].mean()
    s = grp['text_risk_score'].std()
    grade_norm_stats[grade] = (m, s if s > 0.01 else 1.0)

def _normalize(score, grade):
    if grade not in grade_norm_stats:
        return 0.5
    m, s = grade_norm_stats[grade]
    z = (score - m) / s
    return round(min(max(0.5 + z * 0.15, 0.05), 0.95), 4)

subset_train['text_risk_score'] = subset_train.apply(
    lambda r: _normalize(r['text_risk_score'], r['grade']), axis=1)
subset_test['text_risk_score'] = subset_test.apply(
    lambda r: _normalize(r['text_risk_score'], r['grade']), axis=1)

print(f"Normalized text_risk_score (train): mean={subset_train['text_risk_score'].mean():.3f}, "
      f"std={subset_train['text_risk_score'].std():.3f}, "
      f">0.5: {(subset_train['text_risk_score']>0.5).mean():.1%}")

# text_stats from TRAIN only (no test leakage)
text_stats = {}
for grade, grp in subset_train.groupby('grade'):
    text_stats[grade] = {
        'text_risk1_rate':     round(float((grp['text_risk_score'] > 0.5).mean()), 3),
        'actual_default_rate': round(float(grp['label'].mean()), 3),
        'avg_confidence':      round(float(grp['text_confidence'].mean()), 3),
        'avg_risk_score':      round(float(grp['text_risk_score'].mean()), 3),
        'n':                   len(grp),
    }
print("\nPer-grade text stats (train, normalized):")
print(pd.DataFrame(text_stats).T)


subset_train: 552 records, default rate: 0.24
subset_test:  34  records, default rate: 0.26
Raw text_risk_score (train): mean=0.312, std=0.086
Normalized text_risk_score (train): mean=0.498, std=0.142, >0.5: 28.1%

Per-grade text stats (train, normalized):
   text_risk1_rate  actual_default_rate  avg_confidence  avg_risk_score      n
C            0.198                0.201           0.675           0.498  283.0
D            0.281                0.231           0.668           0.497  160.0
E            0.582                0.328           0.699           0.499   67.0
F            0.306                0.444           0.693           0.500   36.0
G            0.667                0.000           0.713           0.500    6.0


---
## Multi-Agent System

**Green Agent** owns the quantitative pipeline: trains XGBoost, evaluates fusion, PSI drift detection, precision gate. White Agents cannot override these decisions.

**Five White Agents** — the system shifts the narrative from *predicting better* to *when to trust the text*:

| # | Agent | LLM policy |
|---|---|---|
| 1 | Text Analyst | Always — free text requires LLM interpretation |
| 2 | Feature Strategist | Conditional — only when grade signals are mixed/ambiguous |
| 3 | Subgroup Advocate | Never — purely quantitative checks (AUC, precision, recall) |
| 4 | Arbitrator | On-demand — only when Advocate vetoes a Strategist proposal |
| 5 | Reporter | Per fuzzy-zone case — explains and audits final predictions |

**Iterative loop flow:**  
Green evaluates → Advocate checks → *if veto*: Strategist proposes + Arbitrator mediates → *else*: Strategist updates directly → repeat until convergence

#### Green Agent

In [113]:
def call_llm(prompt, max_tokens=300, temperature=0.2):
    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": max_tokens, "temperature": temperature},
        timeout=30,
    )
    return resp.json()['choices'][0]['message']['content']

class GreenAgent:
    """
    Execution environment. Owns: baseline model, subgroup diagnostics,
    PSI shift detection, fusion evaluation, threshold calibration, precision gate.
    White Agents cannot override any of these decisions.
    """

    def __init__(self, trained_model, num_features):
        self.model        = trained_model
        self.num_features = num_features

    def run_diagnostics(self, test_df, y_test) -> dict:
        preds = self.model.predict_proba(test_df[self.num_features].fillna(0))[:, 1]
        diag  = test_df.copy()
        diag['_pred']  = preds
        diag['_label'] = y_test.values
        diag['_error'] = ((preds > 0.5).astype(int) != y_test.values).astype(int)
        overall_error  = diag['_error'].mean()

        grade_stats = {}
        for grade, grp in diag.groupby('grade'):
            y_g = grp['_label']
            auc = round(float(roc_auc_score(y_g, grp['_pred'])), 3) if y_g.nunique() > 1 else float('nan')
            grade_stats[grade] = {
                'n':            len(grp),
                'default_rate': round(float(y_g.mean()), 3),
                'error_rate':   round(float(grp['_error'].mean()), 3),
                'auc':          auc,
            }
        high_error = [
            g for g, s in grade_stats.items()
            if not np.isnan(s['error_rate'])
            and s['error_rate'] - overall_error > 0.05
            and s['n'] >= 10
        ]
        return {
            'baseline_preds':    preds,
            'overall_auc':       round(float(roc_auc_score(y_test, preds)), 4),
            'overall_error':     round(float(overall_error), 3),
            'grade_stats':       grade_stats,
            'high_error_grades': high_error,
            'interruption':      f"Subgroup error spike: {high_error}" if high_error else None,
        }

    def evaluate_fusion(self, subset_df, strategy) -> dict:
        """
        Evaluates a fusion strategy on subset_df.
        Returns per-grade AUC, precision, and recall — the single source of truth
        for all White Agents (Advocate, Arbitrator, Reporter).
        """
        from sklearn.metrics import precision_score as ps, recall_score as rs

        grade_weights  = strategy.get('grade_weights', {})
        conf_threshold = strategy.get('conf_threshold', 0.65)

        df_r           = subset_df.reset_index(drop=True)
        baseline_preds = self.model.predict_proba(df_r[self.num_features].fillna(0))[:, 1]

        text_weight = df_r['grade'].map(grade_weights).fillna(0.0).values.astype(float).copy()
        text_weight[df_r['text_confidence'].values < conf_threshold] = 0.0

        # Logit-space fusion: fused_logit = logit(baseline) + α × conf × [logit(text) − logit(0.5)]
        text_evidence = logit(df_r['text_risk_score'].values) - logit(0.5)
        fused_logit   = logit(baseline_preds) + text_weight * df_r['text_confidence'].values * text_evidence
        fused_preds   = np.where(text_weight == 0, baseline_preds, sigmoid(fused_logit))
        y_true      = df_r['label'].values

        grade_auc = {}
        for grade in sorted(df_r['grade'].unique()):
            mask = (df_r['grade'] == grade).values
            y_g  = y_true[mask]
            if mask.sum() < 10 or len(np.unique(y_g)) < 2:
                continue
            b      = roc_auc_score(y_g, baseline_preds[mask])
            f      = roc_auc_score(y_g, fused_preds[mask])
            f_lbl  = (fused_preds[mask] > 0.5).astype(int)
            grade_auc[grade] = {
                'baseline':  round(b, 4),
                'fused':     round(f, 4),
                'delta':     round(f - b, 4),
                'precision': round(ps(y_g, f_lbl, zero_division=0), 4),
                'recall':    round(rs(y_g, f_lbl, zero_division=0), 4),
                'n':         int(mask.sum()),
            }

        return {
            'baseline_preds':       baseline_preds,
            'fused_preds':          fused_preds,
            'overall_baseline_auc': round(float(roc_auc_score(y_true, baseline_preds)), 4),
            'overall_fused_auc':    round(float(roc_auc_score(y_true, fused_preds)), 4),
            'grade_auc':            grade_auc,
            'strategy_applied':     strategy,
        }


    def compute_flip_stats(self, baseline_preds, fused_preds, y_true, threshold=0.5) -> dict:
        """
        Analyzes verdict flips caused by text fusion.
        raised  = baseline says no-default, fused says default
        lowered = baseline says default,    fused says no-default
        """
        b = (baseline_preds >= threshold).astype(int)
        f = (fused_preds    >= threshold).astype(int)
        flipped = b != f
        stats   = {'n_flips': int(flipped.sum()),
                   'flip_rate': round(float(flipped.mean()), 4)}
        raised  = flipped & (f > b)
        lowered = flipped & (f < b)
        if raised.sum()  > 0:
            stats['raised_n']         = int(raised.sum())
            stats['raised_precision'] = round(float(y_true[raised].mean()), 4)
        if lowered.sum() > 0:
            stats['lowered_n']        = int(lowered.sum())
            stats['lowered_recall']   = round(float((1 - y_true[lowered]).mean()), 4)
        return stats

    def validate_strategy(self, subset_df, strategy,
                          max_mean_shift=0.08, min_flip_precision=0.45) -> dict:
        """
        Green's independent multi-criteria validation of a proposed fusion strategy.
        Checks: (1) shift magnitude, (2) raised-verdict precision, (3) lowered-verdict recall.
        Returns passed=False if any hard criterion is violated.
        """
        df_r     = subset_df.reset_index(drop=True)
        baseline = self.model.predict_proba(df_r[self.num_features].fillna(0))[:, 1]
        gw       = strategy.get('grade_weights', {})
        thr      = strategy.get('conf_threshold', 0.65)
        w        = df_r['grade'].map(gw).fillna(0.0).values.astype(float)
        w[df_r['text_confidence'].values < thr] = 0.0
        text_ev  = logit(df_r['text_risk_score'].values) - logit(0.5)
        f_logit  = logit(baseline) + w * df_r['text_confidence'].values * text_ev
        fused    = np.where(w == 0, baseline, sigmoid(f_logit))
        y        = df_r['label'].values

        issues      = []
        passed      = True
        mean_shift  = float(np.abs(fused - baseline).mean())

        if mean_shift > max_mean_shift:
            issues.append(f"Mean shift {mean_shift:.3f} > {max_mean_shift}: text dominating numeric model")
            passed = False

        flip_stats = self.compute_flip_stats(baseline, fused, y)
        if flip_stats['n_flips'] > 0:
            rp = flip_stats.get('raised_precision', 1.0)
            lr = flip_stats.get('lowered_recall',   1.0)
            if rp < min_flip_precision:
                issues.append(f"Raised-flip precision {rp:.3f} < {min_flip_precision}: too many false alarms")
                passed = False
            if lr < min_flip_precision:
                issues.append(f"Lowered-flip recall {lr:.3f} < {min_flip_precision}: too many missed defaults")
                passed = False

        return {'passed': passed, 'issues': issues,
                'mean_shift': round(mean_shift, 4), 'flip_stats': flip_stats}

    def evaluate_candidates(self, subset_df, strategies: list) -> list:
        """
        Evaluates candidate strategies with multi-criteria ranking.
        Primary:   strategies that pass Green validation (shift + flip quality)
        Secondary: overall fused AUC among passing strategies
        Failing strategies are ranked last regardless of AUC.
        """
        results = []
        for strat in strategies:
            r   = self.evaluate_fusion(subset_df, strat)
            val = self.validate_strategy(subset_df, strat)
            results.append({
                'strategy':     strat,
                'fused_auc':    r['overall_fused_auc'],
                'baseline_auc': r['overall_baseline_auc'],
                'grade_auc':    r['grade_auc'],
                'result':       r,
                'passed':       val['passed'],
                'mean_shift':   val['mean_shift'],
                'flip_stats':   val['flip_stats'],
                'issues':       val['issues'],
            })
        # Passing strategies ranked by AUC first; failing strategies ranked last
        return sorted(results, key=lambda x: (not x['passed'], -x['fused_auc']))


    def green_llm_review(self, candidate_results: list, subset_df, iteration: int) -> dict:
        """
        Called when evaluate_candidates finds no passing strategy.
        Green LLM reviews the least-bad option and decides:
          accept     — use the best-failing strategy anyway (with warning)
          adjust     — halve weights for failing grades
          safe_reset — reset all weights to a conservative baseline
        Returns a strategy dict with Green's final decision.
        """
        best = candidate_results[0]
        issues_summary = "; ".join(best.get("issues", ["unknown issue"]))
        flip_stats     = best.get("flip_stats", {})
        grade_deltas   = {g: round(s["delta"], 4)
                          for g, s in best.get("grade_auc", {}).items()}

        prompt = (
            f"You are the Green Agent — iteration {iteration}.\n"
            "evaluate_candidates found NO strategy that passed all validation criteria.\n"
            "You must make a terminal decision on how to proceed.\n\n"
            f"Best candidate strategy: {json.dumps(best['strategy']['grade_weights'])}\n"
            f"Its fused AUC: {best['fused_auc']:.4f}\n"
            f"Validation issues: {issues_summary}\n"
            f"Mean shift from baseline: {best.get('mean_shift', 0):.4f}\n"
            f"Flip stats: {json.dumps(flip_stats)}\n"
            f"Per-grade AUC deltas: {json.dumps(grade_deltas)}\n\n"
            "Choose one action:\n"
            "  accept     — accept the best-failing strategy (AUC improvement justifies the risk)\n"
            "  adjust     — halve weights for grades with negative or zero delta\n"
            "  safe_reset — reset all weights to 0.05 (conservative)\n\n"
            "Rules:\n"
            "- If mean_shift > 0.12 → prefer safe_reset\n"
            "- If fused AUC > baseline AUC by > 0.01 → consider accept\n"
            "- If flip stats show poor precision (< 0.40) → prefer adjust or safe_reset\n\n"
            'Output only JSON: {"action": "<accept|adjust|safe_reset>", "reasoning": "<one sentence>"}'
        )
        raw = call_llm(prompt, max_tokens=180, temperature=0.2)
        cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip())
        cleaned = re.sub(r"\s*```$", "", cleaned.strip())
        try:
            result = json.loads(re.search(r"\{.*\}", cleaned, re.DOTALL).group(0))
        except Exception as e:
            result = {"action": "safe_reset", "reasoning": f"[parse error: {e}]"}

        action = result.get("action", "safe_reset")
        prev_w = best["strategy"]["grade_weights"]

        if action == "accept":
            final_strategy = best["strategy"]
        elif action == "adjust":
            new_w = {g: (round(w * 0.5, 3) if grade_deltas.get(g, 0) <= 0 else w)
                     for g, w in prev_w.items()}
            final_strategy = {**best["strategy"], "grade_weights": new_w}
        else:  # safe_reset
            final_strategy = {**best["strategy"],
                               "grade_weights": {g: 0.05 for g in prev_w}}

        return {
            "strategy":  final_strategy,
            "action":    action,
            "reasoning": result.get("reasoning", ""),
            "llm_raw":   raw,
        }

    def check_psi(self, train_df, target_df, bins=10) -> dict:
        psi_results = {}
        for feat in self.num_features:
            try:
                t_vals = train_df[feat].dropna().values
                g_vals = target_df[feat].dropna().values
                bps    = np.unique(np.percentile(t_vals, np.linspace(0, 100, bins + 1)))
                if len(bps) < 3:
                    continue
                t_pct = np.histogram(t_vals, bins=bps)[0].astype(float)
                g_pct = np.histogram(g_vals, bins=bps)[0].astype(float)
                t_pct = t_pct / t_pct.sum() + 1e-8
                g_pct = g_pct / g_pct.sum() + 1e-8
                psi_results[feat] = round(float(np.sum((g_pct - t_pct) * np.log(g_pct / t_pct))), 4)
            except Exception:
                continue
        flagged = {f: v for f, v in psi_results.items() if v > 0.25}
        return {
            'psi_by_feature':   psi_results,
            'flagged_features': flagged,
            'interruption':     f"PSI drift detected: {list(flagged.keys())}" if flagged else None,
        }

    def find_threshold(self, preds, y_true, min_precision=0.40) -> dict:
        """
        Search [0.05, 0.95] for lowest threshold meeting min_precision with highest recall.
        Call on validation data only — never on test set.
        """
        from sklearn.metrics import recall_score
        best = {'threshold': 0.5, 'precision': 0.0, 'recall': 0.0}
        for t in [i / 100 for i in range(5, 96)]:
            pl   = (np.array(preds) >= t).astype(int)
            if pl.sum() == 0:
                continue
            prec = precision_score(y_true, pl, zero_division=0)
            rec  = recall_score(y_true, pl, zero_division=0)
            if prec >= min_precision and rec > best['recall']:
                best = {'threshold': t, 'precision': round(prec, 3), 'recall': round(rec, 3)}
        return best

    def enforce_threshold(self, preds, y_true, threshold=0.5, min_precision=0.40) -> dict:
        """Final precision gate. White Agents cannot override."""
        pred_labels = (np.array(preds) >= threshold).astype(int)
        if pred_labels.sum() == 0:
            return {'passed': False, 'precision': 0.0, 'reason': 'No positive predictions'}
        prec   = precision_score(y_true, pred_labels, zero_division=0)
        passed = bool(prec >= min_precision)
        return {
            'passed':        passed,
            'precision':     round(float(prec), 3),
            'threshold':     threshold,
            'min_precision': min_precision,
            'reason':        None if passed else f"Precision {prec:.3f} < min {min_precision}",
        }

#### White Agent 2 — Feature Strategist
*LLM called only when grade-level signals are ambiguous (mixed deltas or |delta| < 0.005)*

In [114]:
def feature_strategist_agent(green_result: dict, text_stats: dict,
                              prev_strategy: dict, iteration: int) -> dict:
    """
    Proposes updated grade weights and conf_threshold.
    LLM is always called on iteration 0 (cold start) and whenever any grade has
    an uncertain delta (|delta| < 0.01). Rule-based only when all signals are
    unambiguous AND no grade is uncertain.
    """
    grade_auc = green_result['grade_auc']
    cur_thr   = prev_strategy.get('conf_threshold', 0.65)

    # Significant deltas only (filter noise below 0.005)
    sig = {g: s['delta'] for g, s in grade_auc.items() if abs(s['delta']) >= 0.005}
    clear_positive = bool(sig) and all(d > 0 for d in sig.values())
    clear_negative = bool(sig) and all(d < 0 for d in sig.values())

    # Force LLM if: cold start (iter 0) OR any grade is uncertain (|delta| in [-0.01, +0.01])
    uncertain_grades = [g for g, s in grade_auc.items() if abs(s['delta']) < 0.01]
    force_llm = (iteration == 0) or bool(uncertain_grades)

    if (clear_positive or clear_negative) and not force_llm:
        # Unambiguous and no uncertainty — rule-based update, no LLM call
        new_w = {}
        for grade in ['C', 'D', 'E', 'F', 'G']:
            w     = prev_strategy['grade_weights'].get(grade, 0.0)
            delta = grade_auc.get(grade, {}).get('delta', 0)
            # Match threshold to sig filter: any |delta|>=0.005 gets adjusted
            if delta >= 0.005:
                new_w[grade] = round(min(w + 0.05, 0.6), 3)
            elif delta <= -0.005:
                new_w[grade] = round(max(w - 0.05, 0.0), 3)
            else:
                new_w[grade] = w
        label = "all grades benefit" if clear_positive else "all grades harmed"
        return {
            'grade_weights':  new_w,
            'conf_threshold': cur_thr,
            'rationale':      f'Rule-based ({label})',
            'llm_called':     False,
        }

    # Mixed or weak signals — call LLM to reason through the trade-off
    lines_ctx = []
    for grade, s in sorted(grade_auc.items()):
        ts = text_stats.get(grade, {})
        lines_ctx.append(
            f"  Grade {grade}: baseline={s['baseline']:.3f}, fused={s['fused']:.3f}, "
            f"delta={s['delta']:+.3f}, prec={s.get('precision',0):.3f}, "
            f"rec={s.get('recall',0):.3f} | "
            f"avg_text_risk={ts.get('avg_risk_score','n/a')}, "
            f"actual_default={ts.get('actual_default_rate','n/a')}, "
            f"avg_conf={ts.get('avg_confidence','n/a')}, n={ts.get('n','n/a')}"
        )

    prompt = (
        f"You are the Feature Strategist in a loan default prediction system.\n"
        f"Iteration {iteration} — grade signals are mixed (some improve, others degrade):\n\n"
        + "\n".join(lines_ctx)
        + f"\n\nPrevious strategy:\n"
        f"  grade_weights: {json.dumps(prev_strategy.get('grade_weights', {}))}\n"
        f"  conf_threshold: {cur_thr}\n\n"
        "Context:\n"
        "- Text risk scores are z-score normalized per grade (re-centered at 0.5, std≈0.15).\n"
        "- A grade's avg_text_risk close to its actual_default_rate means text signal is well-calibrated.\n"
        "- Do NOT increase a grade's weight if its delta was negative this iteration.\n"
        "- Weights in [0.0, 0.6]. conf_threshold in [0.55, 0.85].\n\n"
        "CRITICAL: Your entire response must be ONLY a JSON object. "  
        "No explanation, no preamble. Start with { and end with }.\n\n"  
        '{"grade_weights": {"C": <float>, "D": <float>, "E": <float>, '
        '"F": <float>, "G": <float>}, '
        '"conf_threshold": <float 0.55-0.85>, '
        '"rationale": "<one sentence>"}'
    )

    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": 200, "temperature": 0.1},
        timeout=30
    )
    raw = resp.json()['choices'][0]['message']['content']
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON: {raw[:200]}")
    result = json.loads(match.group(0))

    # Clamp weights returned by LLM to valid range
    result['grade_weights'] = {
        g: round(max(0.0, min(0.6, float(w))), 3)
        for g, w in result.get('grade_weights', {}).items()
    }
    result['conf_threshold'] = round(
        max(0.55, min(0.85, float(result.get('conf_threshold', cur_thr)))), 2)

    # Sanity check: LLM must not keep or raise weight for grades with negative delta
    for grade, delta in grade_auc.items():
        d = delta['delta']
        prev_w = prev_strategy['grade_weights'].get(grade, 0.0)
        new_w  = result['grade_weights'].get(grade, 0.0)
        if d < -0.005 and new_w >= prev_w and prev_w > 0:
            result['grade_weights'][grade] = round(max(prev_w - 0.05, 0.0), 3)

    result['llm_called'] = True
    return result

#### White Agent 3 — Subgroup Advocate
*Rule-based only — no LLM needed; quantitative checks on AUC, precision, recall per grade*

In [115]:
def subgroup_advocate_agent(prev_result, curr_result,
                            veto_threshold=0.02) -> dict:
    """
    Checks whether the new strategy harms any borrower subgroup.
    Consumes per-grade precision/recall from Green's evaluate_fusion output —
    no raw prediction arrays needed here.
    Pure rule-based: no LLM.
    """
    if prev_result is None:
        return {'veto': False, 'affected_grades': [], 'critique': None, 'constraints': []}

    degraded = []
    for grade, curr in curr_result['grade_auc'].items():
        if grade not in prev_result['grade_auc']:
            continue
        prev = prev_result['grade_auc'][grade]

        auc_drop  = prev['fused'] - curr['fused']
        prec_drop = prev.get('precision', 0) - curr.get('precision', 0)

        issues = []
        if auc_drop  > veto_threshold: issues.append(f"AUC -{auc_drop:.3f}")
        if prec_drop > 0.05:           issues.append(f"precision -{prec_drop:.3f}")

        if issues:
            degraded.append({
                'grade':     grade,
                'auc_drop':  round(auc_drop, 4),
                'prec_drop': round(prec_drop, 4),
                'recall':    curr.get('recall', float('nan')),
                'issues':    issues,
            })

    if not degraded:
        return {'veto': False, 'affected_grades': [], 'critique': None, 'constraints': []}

    affected    = [d['grade'] for d in degraded]
    constraints = [
        f"Grade {d['grade']}: reduce text weight ({', '.join(d['issues'])})"
        for d in degraded
    ]
    critique = (
        f"Strategy harms Grade(s) {', '.join(affected)}: "
        + "; ".join(f"Grade {d['grade']} {' + '.join(d['issues'])}" for d in degraded)
    )
    return {
        'veto':             True,
        'affected_grades':  affected,
        'critique':         critique,
        'constraints':      constraints,
        'degraded_details': degraded,
    }

#### White Agent 4 — Arbitrator
*LLM called on demand — only when Advocate vetoes a Strategist proposal*

In [116]:
def arbitrator_agent(strategist_proposal: dict, advocate_critique: dict,
                     green_result: dict, prev_strategy: dict, iteration: int) -> dict:
    """
    Mediates the conflict between Strategist and Advocate via LLM.
    Produces a compromise policy that preserves overall gains while
    respecting the Advocate's per-grade constraints.
    Called only when Advocate vetoes.
    """
    grade_lines = [
        f"  Grade {g}: baseline={s['baseline']:.3f}, fused={s['fused']:.3f}, delta={s['delta']:+.3f}"
        for g, s in sorted(green_result['grade_auc'].items())
    ]

    prompt = (
        f"You are the Arbitrator in a loan default prediction multi-agent system.\n\n"
        f"Feature Strategist proposed:\n"
        f"  grade_weights: {json.dumps(strategist_proposal.get('grade_weights', {}))}\n"
        f"  conf_threshold: {strategist_proposal.get('conf_threshold', 0.65)}\n"
        f"  rationale: {strategist_proposal.get('rationale', '')}\n\n"
        f"Subgroup Advocate vetoed:\n"
        f"  {advocate_critique.get('critique', '')}\n"
        f"  Constraints: {'; '.join(advocate_critique.get('constraints', []))}\n\n"
        f"Current performance:\n" + "\n".join(grade_lines) + "\n\n"
        f"Previous strategy:\n"
        f"  grade_weights: {json.dumps(prev_strategy.get('grade_weights', {}))}\n"
        f"  conf_threshold: {prev_strategy.get('conf_threshold', 0.65)}\n\n"
        "Find a compromise: preserve overall AUC gains, reduce (not zero-out) affected grades' weights.\n"
        "Weights in [0.0, 0.6]. conf_threshold in [0.55, 0.85].\n\n"
        "CRITICAL: Your entire response must be ONLY a JSON object. "
        "No explanation, no preamble. Start with { and end with }.\n\n"
        '{"grade_weights": {"C": <float>, "D": <float>, "E": <float>, '
        '"F": <float>, "G": <float>}, '
        '"conf_threshold": <float 0.55-0.85>, '
        '"resolution": "<one sentence>"}'
    )

    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": 250, "temperature": 0.1},
        timeout=30
    )
    raw = resp.json()['choices'][0]['message']['content']
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        raise ValueError(f"Arbitrator: no JSON: {raw[:200]}")
    result = json.loads(match.group(0))
    result['grade_weights'] = {
        g: round(max(0.0, min(0.6, float(w))), 3)
        for g, w in result.get('grade_weights', {}).items()
    }
    result['conf_threshold'] = round(
        max(0.55, min(0.85, float(result.get('conf_threshold', 0.65)))), 2)
    return result

#### Iterative Loop Orchestrator
*Strategist → Advocate → Arbitrator (on conflict) → repeat*

In [117]:
def run_iterative_loop(green, subset_df, text_stats, max_iter=5, veto_threshold=0.02):
    """
    White proposes → Green evaluates → Advocate checks → Arbitrator mediates (on conflict).
    After Arbitrator, Green numerically verifies the compromise vs the simple fallback
    and picks whichever produces higher fused AUC on the training subset.
    """
    strategy = {
        'grade_weights':  {'C': 0.03, 'D': 0.10, 'E': 0.05, 'F': 0.03, 'G': 0.03},
        'conf_threshold': 0.65,
    }
    decision_log = []
    prev_result  = None
    llm_calls    = 0

    for i in range(max_iter):
        print(f"\n{'='*65}")
        print(f"Iter {i}  weights={strategy['grade_weights']}  conf={strategy['conf_threshold']}")

        curr_result = green.evaluate_fusion(subset_df, strategy)
        print(f"  overall  baseline={curr_result['overall_baseline_auc']:.4f}  "
              f"fused={curr_result['overall_fused_auc']:.4f}")
        for grade, s in sorted(curr_result['grade_auc'].items()):
            arrow = '↑' if s['delta'] > 0 else '↓'
            print(f"  Grade {grade}: {s['baseline']:.3f} → {s['fused']:.3f} "
                  f"{arrow}{abs(s['delta']):.3f}  "
                  f"prec={s['precision']:.3f}  rec={s['recall']:.3f}")

        advocate = subgroup_advocate_agent(prev_result, curr_result, veto_threshold)
        log_entry = {
            'iteration':        i,
            'strategy':         strategy.copy(),
            'overall_baseline': curr_result['overall_baseline_auc'],
            'overall_fused':    curr_result['overall_fused_auc'],
            'grade_auc':        curr_result['grade_auc'],
            'veto':             advocate['veto'],
            'veto_grades':      advocate['affected_grades'],
        }

        if advocate['veto']:
            print(f"  [ADVOCATE VETO] {advocate['critique']}")
            try:
                strat_prop = feature_strategist_agent(curr_result, text_stats, strategy, i)
                llm_calls += 1 if strat_prop.get('llm_called') else 0
                print(f"  [STRATEGIST] {strat_prop.get('rationale', '')} "
                      f"(LLM={'yes' if strat_prop.get('llm_called') else 'no'})")

                arb = arbitrator_agent(strat_prop, advocate, curr_result, strategy, i)
                llm_calls += 1

                # Build fallback: halve affected grades
                fallback_w = {g: (round(w * 0.5, 3) if g in advocate['affected_grades'] else w)
                              for g, w in strategy['grade_weights'].items()}
                fallback = {'grade_weights': fallback_w,
                            'conf_threshold': strategy['conf_threshold']}

                # Green verifies: arbitrated vs fallback — pick higher fused AUC
                ranked = green.evaluate_candidates(subset_df, [arb, fallback])
                # If no candidate passes Green validation, call Green LLM
                if not ranked[0]['passed']:
                    print(f"  [GREEN LLM] No passing strategy — calling Green LLM review...")
                    green_review = green.green_llm_review(ranked, subset_df, i)
                    llm_calls += 1
                    strategy = green_review['strategy']
                    source   = f"green_llm:{green_review['action']}"
                    print(f"  [GREEN LLM] {green_review['action']} — {green_review['reasoning']}")
                else:
                    winner = ranked[0]
                    strategy = winner['strategy']
                    source   = 'arbitrated' if winner['strategy'] == arb else 'fallback'
                    print(f"  [ARBITRATOR] {arb.get('resolution', '')}  "
                          f"→ Green verified: {source} wins "
                          f"(AUC={winner['fused_auc']:.4f} vs other={ranked[1]['fused_auc']:.4f})")

                log_entry['event']                 = source
                log_entry['strategist_rationale']  = strat_prop.get('rationale', '')
                log_entry['arbitrator_resolution'] = arb.get('resolution', '')
            except Exception as e:
                print(f"  [ARBITRATOR ERROR] {e} — halving affected weights")
                for grade in advocate['affected_grades']:
                    if grade in strategy['grade_weights']:
                        strategy['grade_weights'][grade] = round(
                            strategy['grade_weights'][grade] * 0.5, 3)
                log_entry['event'] = 'veto_fallback'

            log_entry['veto_critique'] = advocate['critique']
            decision_log.append(log_entry)
            prev_result = curr_result
            continue

        # Convergence: only after a genuine strategy update or arbitration
        last_event = decision_log[-1].get('event', '') if decision_log else ''
        if (prev_result is not None
                and last_event in ('strategy_update', 'arbitrated', 'fallback')  # 'strategy_unchanged' excluded
                and abs(curr_result['overall_fused_auc'] - prev_result['overall_fused_auc']) < 0.002):
            log_entry['event'] = 'converged'
            decision_log.append(log_entry)
            print(f"  Converged at iteration {i}.")
            break

        # No veto — Strategist proposes directly
        try:
            new_strat  = feature_strategist_agent(curr_result, text_stats, strategy, i)
            llm_called = new_strat.get('llm_called', True)
            llm_calls += 1 if llm_called else 0
            new_strategy = {
                'grade_weights':  new_strat['grade_weights'],
                'conf_threshold': new_strat.get('conf_threshold', strategy['conf_threshold']),
            }
            # Detect unchanged strategy: don't mark as 'strategy_update' if nothing changed
            # (prevents false convergence on the next iteration)
            if (new_strategy['grade_weights'] == strategy['grade_weights'] and
                    new_strategy['conf_threshold'] == strategy['conf_threshold']):
                event_label = 'strategy_unchanged'
            else:
                event_label = 'strategy_update'
            strategy = new_strategy
            print(f"  [STRATEGIST→{'LLM' if llm_called else 'RULE'}] {new_strat.get('rationale', '')} [{event_label}]")
            log_entry['event']               = event_label
            log_entry['strategist_rationale'] = new_strat.get('rationale', '')
            log_entry['llm_called']          = llm_called
        except Exception as e:
            print(f"  [STRATEGIST ERROR] {e} — keeping strategy")
            log_entry['event'] = 'strategist_error'

        decision_log.append(log_entry)
        prev_result = curr_result

    print(f"\nLLM calls during loop: {llm_calls}")
    return decision_log, curr_result

#### White Agent 5 — Reporter
*LLM called per prediction when baseline is in fuzzy zone [0.30–0.65] or text significantly shifted the outcome*

In [118]:
def reporter_agent(row: pd.Series, baseline_pred: float, fused_pred: float,
                   strategy: dict) -> dict:
    """
    Generates a faithful, auditable explanation for a single prediction.
    Flags fragile predictions where text overrode a confident numeric signal.
    Called for fuzzy-zone cases (baseline 0.30-0.65) or when text shifted prediction > 0.10.
    """
    eff_weight     = strategy['grade_weights'].get(row['grade'], 0.0)
    conf           = row.get('text_confidence', 0)
    if conf < strategy.get('conf_threshold', 0.65):
        eff_weight = 0.0

    text_influence = fused_pred - baseline_pred
    in_fuzzy_zone  = 0.30 <= baseline_pred <= 0.65
    text_flipped   = (fused_pred > 0.5) != (baseline_pred > 0.5)
    fragile        = text_flipped and abs(text_influence) < 0.15

    prompt = (
        f"You are explaining a loan default prediction to an auditor.\n\n"
        f"Borrower description: {str(row.get('desc', ''))[:400]}\n\n"
        f"Numeric model (XGBoost) prediction: {baseline_pred:.3f}\n"
        f"Text risk score: {row.get('text_risk_score', 'N/A')}\n"
        f"Text confidence: {conf:.3f}\n"
        f"Effective text weight for Grade {row['grade']}: {eff_weight:.2f}\n"
        f"Final fused prediction: {fused_pred:.3f}  (text shifted by {text_influence:+.3f})\n\n"
        f"Key numeric features: int_rate={row.get('int_rate','N/A')}, "
        f"dti={row.get('dti','N/A')}, fico={row.get('fico_range_low','N/A')}, "
        f"annual_inc={row.get('annual_inc','N/A')}\n\n"
        'Output a single JSON object and nothing else:\n'
        '{"explanation": "2-3 sentences describing why this prediction was made",'
        ' "main_driver": "text | numeric | both",'
        ' "faithfulness": "was the prediction mainly moved by text or numeric features?",'
        ' "fragility_flag": true/false,'
        ' "fragility_reason": "null or brief explanation"}'
    )

    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": 300, "temperature": 0.3},
        timeout=30
    )
    raw = resp.json()['choices'][0]['message']['content']
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return {'explanation': 'N/A', 'fragility_flag': fragile,
                'fragility_reason': 'computed', '_error': raw[:100]}
    result = json.loads(match.group(0))
    result['_text_influence'] = round(text_influence, 4)
    result['_in_fuzzy_zone']  = in_fuzzy_zone
    result['_fragile_computed'] = fragile
    return result

#### Run the Full System

In [119]:
# ── Train XGBoost baseline ────────────────────────────────────────────
model = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    scale_pos_weight=(train['label'] == 0).sum() / (train['label'] == 1).sum(),
    random_state=42, eval_metric='auc'
)
model.fit(train[NUM_FEATURES], train['label'])
print(f"Baseline AUC (full test): "
      f"{roc_auc_score(test['label'], model.predict_proba(test[NUM_FEATURES].fillna(0))[:,1]):.4f}")

green = GreenAgent(model, NUM_FEATURES)

baseline_diag = green.run_diagnostics(test, test['label'])
if baseline_diag['interruption']:
    print(f"[GREEN INTERRUPT] {baseline_diag['interruption']}")

psi = green.check_psi(train, subset_train)
if psi['interruption']:
    print(f"[GREEN PSI] {psi['interruption']}")
    print(f"  (Note: C-G grades naturally differ from full train — some drift is expected)")
else:
    print("PSI check: no significant feature drift.")

# ── Iterative loop on TRAIN (tune weights, no test leakage) ───────────
print("\n--- Tuning fusion weights on TRAIN ---")
decision_log, train_result = run_iterative_loop(
    green, subset_train, text_stats, max_iter=5, veto_threshold=0.02
)
best_strategy = decision_log[-1]['strategy']
print(f"\nBest strategy: {best_strategy}")

# ── Threshold calibration on TRAIN (no test leakage) ──────────────────
cal = green.find_threshold(train_result['fused_preds'], subset_train['label'].values,
                           min_precision=0.40)
best_threshold = cal['threshold'] if cal['recall'] > 0 else 0.5
print(f"\nCalibrated threshold (train): {best_threshold}  "
      f"(precision={cal['precision']}, recall={cal['recall']})")

# ── Final evaluation on TEST (held-out, run once) ─────────────────────
print("\n--- Final evaluation on TEST ---")
final_result = green.evaluate_fusion(subset_test, best_strategy)
print(f"Test baseline AUC : {final_result['overall_baseline_auc']:.4f}")
print(f"Test fused   AUC  : {final_result['overall_fused_auc']:.4f}")
for grade, s in sorted(final_result['grade_auc'].items()):
    arrow = '↑' if s['delta'] > 0 else '↓'
    print(f"  Grade {grade}: {s['baseline']:.3f} → {s['fused']:.3f} {arrow}{abs(s['delta']):.3f}")

gate = green.enforce_threshold(
    final_result['fused_preds'], subset_test['label'].values,
    threshold=best_threshold, min_precision=0.40
)
print(f"\nPrecision gate: {'PASSED' if gate['passed'] else 'FAILED'} "
      f"(threshold={gate['threshold']}, precision={gate['precision']}, "
      f"min_required={gate['min_precision']})")

Baseline AUC (full test): 0.6881
[GREEN INTERRUPT] Subgroup error spike: ['C', 'D', 'E', 'F', 'G']
[GREEN PSI] PSI drift detected: ['int_rate', 'fico_range_low', 'fico_range_high', 'revol_util']
  (Note: C-G grades naturally differ from full train — some drift is expected)

--- Tuning fusion weights on TRAIN ---

Iter 0  weights={'C': 0.03, 'D': 0.1, 'E': 0.05, 'F': 0.03, 'G': 0.03}  conf=0.65
  overall  baseline=0.7408  fused=0.7439
  Grade C: 0.762 → 0.764 ↑0.001  prec=0.302  rec=0.860
  Grade D: 0.723 → 0.734 ↑0.011  prec=0.276  rec=0.946
  Grade E: 0.688 → 0.691 ↑0.003  prec=0.349  rec=1.000
  Grade F: 0.738 → 0.741 ↑0.003  prec=0.471  rec=1.000
  [STRATEGIST→LLM] All grades showed positive deltas, but the improvements were marginal, so we maintain the previous weights. [strategy_unchanged]

Iter 1  weights={'C': 0.03, 'D': 0.1, 'E': 0.05, 'F': 0.03, 'G': 0.03}  conf=0.65
  overall  baseline=0.7408  fused=0.7439
  Grade C: 0.762 → 0.764 ↑0.001  prec=0.302  rec=0.860
  Grade D: 0.72

In [ ]:
# ── Save model artifacts to pkl ───────────────────────────────────────────────
import pickle, os

# Build grade_train_stats: per-grade AUC delta at learned weight and at α=0.20
grade_train_stats = {}
for grade in ['C', 'D', 'E', 'F', 'G']:
    learned_w = best_strategy['grade_weights'].get(grade, 0.0)
    s = train_result['grade_auc'].get(grade, {})
    delta_learned = s.get('delta', 0.0)

    # Measure AUC delta at high weight (α=0.20) for Advocate reference in demo
    high_strat = {**best_strategy, 'grade_weights': {**best_strategy['grade_weights'], grade: 0.20}}
    high_res   = green.evaluate_fusion(subset_train, high_strat)
    delta_high = high_res['grade_auc'].get(grade, {}).get('delta', 0.0)

    if delta_learned <= -0.005:
        reason = 'text fusion hurt AUC at any weight'
    elif delta_learned > 0.01:
        reason = 'text signal improves AUC'
    elif learned_w == 0.0:
        reason = 'insufficient samples or no signal'
    else:
        reason = 'marginal, small weight only'

    grade_train_stats[grade] = {
        'delta_at_learned_weight': round(delta_learned, 4),
        'delta_at_high_weight':    round(delta_high, 4),
        'learned_weight':          learned_w,
        'reason':                  reason,
    }

print("grade_train_stats:")
for g, v in grade_train_stats.items():
    print(f"  {g}: {v}")

# Save
save_path = os.path.expanduser('~/Desktop/loan_default_model.pkl')
with open(save_path, 'wb') as f:
    pickle.dump({
        'model':             model,
        'features':          NUM_FEATURES,
        'best_strategy':     best_strategy,
        'grade_norm_stats':  grade_norm_stats,
        'grade_train_stats': grade_train_stats,
    }, f)
print(f"\nSaved → {save_path}")


#### Decision Log

In [120]:
rows = []
for e in decision_log:
    rows.append({
        'iter':         e['iteration'],
        'event':        e.get('event', '-'),
        'baseline_auc': e['overall_baseline'],
        'fused_auc':    e['overall_fused'],
        'delta':        round(e['overall_fused'] - e['overall_baseline'], 4),
        'veto':         e['veto'],
        'veto_grades':  ','.join(e['veto_grades']) if e['veto_grades'] else '-',
        'llm':          'yes' if e.get('llm_called') else ('arb' if e.get('event') == 'arbitrated' else 'no'),
        'conf_thr':     e['strategy']['conf_threshold'],
        'w_C':          e['strategy']['grade_weights'].get('C', 0),
        'w_D':          e['strategy']['grade_weights'].get('D', 0),
        'w_E':          e['strategy']['grade_weights'].get('E', 0),
        'note':         e.get('arbitrator_resolution',
                         e.get('strategist_rationale',
                          e.get('veto_critique', '-')))[:60],
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== Grade AUC Trajectory ===")
grade_rows = []
for e in decision_log:
    for grade, s in e['grade_auc'].items():
        grade_rows.append({'iter': e['iteration'], 'grade': grade,
                           'baseline': s['baseline'], 'fused': s['fused'],
                           'delta': s['delta']})
print(pd.DataFrame(grade_rows).to_string(index=False))

 iter              event  baseline_auc  fused_auc  delta  veto veto_grades llm  conf_thr  w_C  w_D  w_E                                                         note
    0 strategy_unchanged        0.7408     0.7439 0.0031 False           - yes      0.65 0.03 0.10 0.05 All grades showed positive deltas, but the improvements were
    1 strategy_unchanged        0.7408     0.7439 0.0031 False           - yes      0.65 0.03 0.10 0.05 All grades showed improvements (positive deltas), but the ch
    2 strategy_unchanged        0.7408     0.7439 0.0031 False           - yes      0.65 0.03 0.10 0.05 Grades C, D, E, and F showed improvements (positive deltas),
    3    strategy_update        0.7408     0.7439 0.0031 False           - yes      0.65 0.03 0.10 0.05 Grades D and E showed improvements and have higher actual de
    4          converged        0.7408     0.7447 0.0039 False           -  no      0.65 0.03 0.12 0.06                                                            -

=== Grade

#### Comparison 1 — Meta-Model (Text as Features)

In [121]:
from sklearn.linear_model import LogisticRegression

def add_text_flags(df):
    """Regex-based binary flags — no extra API calls needed."""
    d = df.copy()
    desc = d['desc'].fillna('').str.lower()
    d['flag_consolidation'] = desc.str.contains('consolidat').astype(float)
    d['flag_repayment_plan'] = desc.str.contains(r'will pay|plan to|repay|pay off', regex=True).astype(float)
    d['flag_stress']        = desc.str.contains(r'behind|urgent|emergency|struggling|desperate', regex=True).astype(float)
    d['flag_stable_income'] = desc.str.contains(r'stable|steady|permanent|full.time|full time', regex=True).astype(float)
    return d

META_FEATURES = NUM_FEATURES + ['text_risk_score', 'text_confidence',
                                  'flag_consolidation', 'flag_repayment_plan',
                                  'flag_stress', 'flag_stable_income']

tr = add_text_flags(subset_train)
te = add_text_flags(subset_test)

meta_model = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1)
meta_model.fit(tr[META_FEATURES].fillna(0), tr['label'])

meta_preds = meta_model.predict_proba(te[META_FEATURES].fillna(0))[:, 1]
print(f"Baseline (numeric only) AUC : {final_result['overall_baseline_auc']:.4f}")
print(f"Weighted fusion AUC          : {final_result['overall_fused_auc']:.4f}")
print(f"Meta-model (LR stacking) AUC : {roc_auc_score(subset_test['label'], meta_preds):.4f}")

# Show per-grade meta-model AUC
print("\nPer-grade meta-model AUC:")
for grade in sorted(te['grade'].unique()):
    mask = (te['grade'] == grade).values
    y_g  = subset_test['label'].values[mask]
    if mask.sum() < 10 or len(np.unique(y_g)) < 2:
        continue
    b = roc_auc_score(y_g, final_result['baseline_preds'][mask])
    m = roc_auc_score(y_g, meta_preds[mask])
    print(f"  Grade {grade}: baseline={b:.3f}  meta={m:.3f}  delta={m-b:+.3f}")

Baseline (numeric only) AUC : 0.6444
Weighted fusion AUC          : 0.6444
Meta-model (LR stacking) AUC : 0.5733

Per-grade meta-model AUC:
  Grade C: baseline=0.738  meta=0.750  delta=+0.012
  Grade D: baseline=0.667  meta=0.458  delta=-0.208


/Users/ljw/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### Comparison 2 — Grid Search vs Strategist

In [122]:
import itertools

# Grid search on TRAIN subset — all combinations
cands_c   = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
cands_d   = [0.0, 0.05, 0.1, 0.15, 0.2]
cands_e   = [0.0, 0.05, 0.1]
cands_thr = [0.55, 0.60, 0.65, 0.70, 0.75]

best_grid_auc  = -1
best_grid_strat = None

for w_c, w_d, w_e, thr in itertools.product(cands_c, cands_d, cands_e, cands_thr):
    strat = {'grade_weights': {'C': w_c, 'D': w_d, 'E': w_e, 'F': 0.0, 'G': 0.0},
             'conf_threshold': thr}
    res = green.evaluate_fusion(subset_train, strat)
    if res['overall_fused_auc'] > best_grid_auc:
        best_grid_auc   = res['overall_fused_auc']
        best_grid_strat = strat

print(f"Grid search best (train AUC={best_grid_auc:.4f}): {best_grid_strat}")

grid_test = green.evaluate_fusion(subset_test, best_grid_strat)
strat_test_auc = final_result['overall_fused_auc']

print(f"\nTest AUC comparison:")
print(f"  Baseline                  : {final_result['overall_baseline_auc']:.4f}")
print(f"  Strategist (MAS loop)     : {strat_test_auc:.4f}")
print(f"  Grid search (450 combos)  : {grid_test['overall_fused_auc']:.4f}")
print(f"  Strategist vs grid search : {strat_test_auc - grid_test['overall_fused_auc']:+.4f}")

Grid search best (train AUC=0.7473): {'grade_weights': {'C': 0.0, 'D': 0.2, 'E': 0.1, 'F': 0.0, 'G': 0.0}, 'conf_threshold': 0.6}

Test AUC comparison:
  Baseline                  : 0.6444
  Strategist (MAS loop)     : 0.6444
  Grid search (450 combos)  : 0.6178
  Strategist vs grid search : +0.0266


#### Live Walkthrough — Individual Case Analysis

In [123]:
def show_walkthrough(subset_df, green, final_strategy, n_cases=4, call_reporter=True):
    """
    Displays representative prediction cases.
    For each case, optionally calls Reporter Agent for faithful explanation + fragility audit.
    """
    df_w = subset_df.copy().reset_index(drop=True)
    df_w['baseline_pred'] = green.model.predict_proba(df_w[NUM_FEATURES].fillna(0))[:, 1]

    gw  = final_strategy['grade_weights']
    thr = final_strategy['conf_threshold']
    df_w['eff_weight'] = df_w['grade'].map(gw).fillna(0.0)
    df_w.loc[df_w['text_confidence'] < thr, 'eff_weight'] = 0.0
    # Logit-space fusion (mirrors evaluate_fusion)
    _text_ev = logit(df_w['text_risk_score'].values) - logit(0.5)
    _fused_logit = logit(df_w['baseline_pred'].values) + df_w['eff_weight'].values * df_w['text_confidence'].values * _text_ev
    df_w['fused_pred'] = np.where(df_w['eff_weight'] == 0, df_w['baseline_pred'],
                                  sigmoid(_fused_logit))
    df_w['baseline_ok'] = ((df_w['baseline_pred'] > 0.5) == df_w['label'])
    df_w['fused_ok']    = ((df_w['fused_pred']    > 0.5) == df_w['label'])
    df_w['disagree']    = abs(df_w['baseline_pred'] - df_w['text_risk_score'])
    df_w['in_fuzzy']    = (df_w['baseline_pred'].between(0.30, 0.65))

    case_pools = [
        ("Text HELPED  — baseline wrong, fusion right",
         df_w[~df_w['baseline_ok'] & df_w['fused_ok'] & (df_w['eff_weight'] > 0)]),
        ("Text HURT    — baseline right, fusion wrong",
         df_w[df_w['baseline_ok'] & ~df_w['fused_ok'] & (df_w['eff_weight'] > 0)]),
        ("FUZZY ZONE   — numeric model uncertain (baseline 0.30-0.65)",
         df_w[df_w['in_fuzzy'] & (df_w['eff_weight'] > 0)].nlargest(5, 'disagree')),
        ("Text SUPPRESSED — confidence below threshold",
         df_w[df_w['eff_weight'] == 0].nlargest(5, 'text_confidence')),
    ]

    shown = 0
    for case_label, pool in case_pools:
        if shown >= n_cases or len(pool) == 0:
            continue
        row = pool.iloc[0]
        shown += 1

        print(f"\n{'='*70}")
        print(f"CASE: {case_label}")
        print(f"Grade: {row['grade']}  |  Actual: {'DEFAULT' if row['label']==1 else 'FULLY PAID'}")
        print(f"{'─'*70}")
        print(f"Description:\n  {str(row['desc'])[:350]}")
        print(f"{'─'*70}")
        print(f"[Text Analyst]   risk_score={row['text_risk_score']:.3f}  "
              f"confidence={row['text_confidence']:.3f}")
        print(f"[Green Numeric]  baseline_pred={row['baseline_pred']:.3f}"
              f"{'  ← fuzzy zone' if row['in_fuzzy'] else ''}")
        print(f"[Fusion]         eff_weight={row['eff_weight']:.2f} (thr={thr})  "
              f"fused_pred={row['fused_pred']:.3f}  "
              f"(text shift {row['fused_pred']-row['baseline_pred']:+.3f})")
        print(f"[Outcome]        {'CORRECT ✓' if row['fused_ok'] else 'WRONG ✗'}")

        if call_reporter and (row['in_fuzzy'] or abs(row['fused_pred'] - row['baseline_pred']) > 0.10):
            try:
                report = reporter_agent(row, row['baseline_pred'], row['fused_pred'], final_strategy)
                print(f"[Reporter]")
                print(f"  Explanation : {report.get('explanation', 'N/A')}")
                print(f"  Main driver : {report.get('main_driver', 'N/A')}")
                print(f"  Faithfulness: {report.get('faithfulness', 'N/A')}")
                if report.get('fragility_flag') or report.get('_fragile_computed'):
                    reason = report.get('fragility_reason') or 'text flipped a borderline numeric prediction'
                    print(f"  ⚠ FRAGILE  : {reason}")
            except Exception as e:
                print(f"  [REPORTER ERROR] {e}")


show_walkthrough(subset_test, green, best_strategy)


CASE: FUZZY ZONE   — numeric model uncertain (baseline 0.30-0.65)
Grade: C  |  Actual: FULLY PAID
──────────────────────────────────────────────────────────────────────
Description:
    Borrower added on 11/07/10 > This will consolidate 4 credit cards into a lower monthly payment. Currently I Pay $250monthly to cover the minimum on these cards.<br/> Borrower added on 11/07/10 > Planning to set up automatic payments from checking to cover monthly payments. Also expect to have this paid off early.  One of my 2 jobs is Residential 
──────────────────────────────────────────────────────────────────────
[Text Analyst]   risk_score=0.201  confidence=0.950
[Green Numeric]  baseline_pred=0.429  ← fuzzy zone
[Fusion]         eff_weight=0.03 (thr=0.65)  fused_pred=0.420  (text shift -0.010)
[Outcome]        CORRECT ✓
[Reporter]
  Explanation : The prediction of a potential loan default is influenced by a moderate interest rate and debt-to-income ratio, combined with an average FICO score and re